Imports


In [1]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML, display


Initial Conditions and Physical Scales

In [2]:
# ============================================================
# CELL 2 — INITIAL CONDITIONS AND PHYSICAL SCALES 
# ============================================================


# ------------------------------------------------------------
# BODY MASSES
# Set each body's mass independently, in kilograms. The array order is used
# throughout the notebook: masses[0] = Body 1, and so on.
# ------------------------------------------------------------

masses = np.array([
    1.0e30,                     # Body 1 (kg)
    1.0e30,                     # Body 2 (kg)
    1.0e30,                     # Body 3 (kg)
], dtype=float)

# ------------------------------------------------------------
# INITIAL POSITIONS
# Each row contains one body's dimensionless (x, y) coordinates.
# Physical coordinates are calculated later by multiplying by LENGTH_SCALE.
# ------------------------------------------------------------

initial_positions = np.array([
    [-1.0, 0.0],                 # Body 1: (x, y) * L_0
    [ 1.0, 0.0],                 # Body 2: (x, y) * L_0
    [ 0.0, 7.5],                 # Body 3: (x, y) * L_0
], dtype=float)


# ------------------------------------------------------------
# INITIAL VELOCITIES
# Each row contains one body's dimensionless (vx, vy) components.
# Physical velocities are calculated later by multiplying by VELOCITY_SCALE.
# ------------------------------------------------------------

initial_velocities = np.array([
    [ 0.25, -0.25],              # Body 1: (vx, vy) * V_0
    [-0.25,  0.25],              # Body 2: (vx, vy) * V_0
    [-0.70,  0.00],              # Body 3: (vx, vy) * V_0
], dtype=float)


# ------------------------------------------------------------
# PHYSICAL CONSTANTS
# ------------------------------------------------------------

G = 6.67430e-11                 # Gravitational constant (m^3 kg^-1 s^-2)


# ------------------------------------------------------------
# CHARACTERISTIC LENGTH SCALE
# One dimensionless position unit represents LENGTH_SCALE meters.
# ------------------------------------------------------------

LENGTH_SCALE = 1.0e7         # Characteristic length, L_0 (m)


# ------------------------------------------------------------
# CHARACTERISTIC GRAVITATIONAL SCALES
# The largest body mass is used as the reference mass. These scales convert
# dimensionless positions, velocities, and time into SI units.
# ------------------------------------------------------------

MASS_SCALE = np.max(masses)
TIME_SCALE = np.sqrt(LENGTH_SCALE**3 / (G * MASS_SCALE))
VELOCITY_SCALE = LENGTH_SCALE / TIME_SCALE


# ------------------------------------------------------------
# VALIDATE INPUTS
# Check dimensions and finite, physically valid values before integration.
# ------------------------------------------------------------

if masses.shape != (3,):
    raise ValueError("masses must contain exactly three values.")
if not np.all(np.isfinite(masses)) or np.any(masses <= 0):
    raise ValueError("All body masses must be finite and greater than zero.")
if initial_positions.shape != (3, 2):
    raise ValueError("initial_positions must have shape (3, 2).")
if initial_velocities.shape != (3, 2):
    raise ValueError("initial_velocities must have shape (3, 2).")
if not np.all(np.isfinite(initial_positions)):
    raise ValueError("Initial positions must contain finite values.")
if not np.all(np.isfinite(initial_velocities)):
    raise ValueError("Initial velocities must contain finite values.")
if not np.isfinite(LENGTH_SCALE) or LENGTH_SCALE <= 0:
    raise ValueError("LENGTH_SCALE must be finite and greater than zero.")


print("PHYSICAL SCALES")
print(f"Reference mass: {MASS_SCALE:.6e} kg")
print(f"Length scale (L_0): {LENGTH_SCALE:.6e} m")
print(f"Time scale: {TIME_SCALE:.6e} s")
print(f"Velocity scale (V_0): {VELOCITY_SCALE:.6e} m/s")



PHYSICAL SCALES
Reference mass: 1.000000e+30 kg
Length scale (L_0): 1.000000e+07 m
Time scale: 3.870768e+00 s
Velocity scale (V_0): 2.583467e+06 m/s


Simulation Parameters

In [3]:
# ============================================================
# CELL 3 — SIMULATION PARAMETERS
# ============================================================


# ------------------------------------------------------------
# SIMULATION DURATION
# simulation_duration is measured in characteristic time units.
# ------------------------------------------------------------

simulation_duration = 250

t_start = 0.0
t_end = simulation_duration * TIME_SCALE


# ------------------------------------------------------------
# VIDEO SETTINGS
# video_duration and fps determine the playback length and frame rate.
# num_frames is the number of samples used in the displayed animation.
# ------------------------------------------------------------

video_duration = 20            # Video duration (seconds)
fps = 60
num_frames = video_duration * fps


# ------------------------------------------------------------
# COLLISION AND EJECTION THRESHOLDS
# A collision is detected when a pair's separation falls below collision_radius.
# Ejection requires a body to be distant and energetically unbound relative to
# the other two bodies.
# ------------------------------------------------------------

collision_radius = 0.05 * LENGTH_SCALE
ejection_radius = 20.0 * LENGTH_SCALE


# ------------------------------------------------------------
# NUMERICAL INTEGRATION SETTINGS
# rtol and atol control the baseline solver accuracy. Smaller values generally
# require more computation. 'tolerances' defines the additional settings used
# in the numerical convergence comparison.
# ------------------------------------------------------------

rtol = 1e-10
atol = 1e-12

tolerances = [
    (1e-7,  1e-9),
    (1e-9,  1e-11),
    (1e-11, 1e-13),
]


print("\nSIMULATION PARAMETERS")
print(f"Physical simulation duration: {t_end:.6e} s")
print(f"Video duration: {video_duration} seconds")
print(f"Total video frames: {num_frames}")
print(f"Collision radius: {collision_radius:.3e} m")
print(f"Ejection radius: {ejection_radius:.3e} m")



SIMULATION PARAMETERS
Physical simulation duration: 9.676920e+02 s
Video duration: 20 seconds
Total video frames: 1200
Collision radius: 5.000e+05 m
Ejection radius: 2.000e+08 m


Convert Initial Conditions to SI Units

In [4]:
# ============================================================
# CELL 4 — CONVERT INITIAL CONDITIONS TO SI UNITS
# ============================================================

# Convert the dimensionless starting positions and velocities entered in Cell 3
# into meters and meters per second, respectively, for the numerical solver.
def make_initial_conditions(initial_positions, initial_velocities):
    """Convert dimensionless initial conditions into SI units."""

    positions_si = LENGTH_SCALE * initial_positions
    velocities_si = VELOCITY_SCALE * initial_velocities

    return positions_si, velocities_si


Newtonian Equations of Motion

In [5]:
# ============================================================
# CELL 5 — NEWTONIAN EQUATIONS OF MOTION
# ============================================================

# The solver state contains six position components followed by six velocity
# components. Return their derivatives: the three velocities followed by the
# gravitational accelerations of the three bodies.
def derivatives(t, state):
    """Return the first-order equations of motion for three bodies."""

    positions = state[:6].reshape(3, 2)
    velocities = state[6:].reshape(3, 2)
    accelerations = np.zeros((3, 2))

    for i in range(3):
        for j in range(3):
            if i == j:
                continue

            displacement = positions[j] - positions[i]
            distance = np.linalg.norm(displacement)

            if distance < 1e-10 * LENGTH_SCALE:
                raise ValueError("Gravitational singularity detected.")

            accelerations[i] += (
                G * masses[j] * displacement / distance**3
            )

    return np.concatenate([velocities.ravel(), accelerations.ravel()])


Collision and Ejection Events

In [6]:
# ============================================================
# CELL 6 — COLLISION AND EJECTION EVENTS
# ============================================================


# Create a terminal event for one body pair. The event function reaches zero
# when the pair's separation equals collision_radius.
def make_collision_event(i, j):
    """Create a terminal event for a collision between bodies i and j."""

    def collision_event(t, state):
        positions = state[:6].reshape(3, 2)
        distance = np.linalg.norm(positions[i] - positions[j])
        return distance - collision_radius

    collision_event.terminal = True
    collision_event.direction = -1
    return collision_event


# Create a terminal event for body i. It triggers only when the body is both
# beyond ejection_radius from the remaining pair's center of mass and has
# positive specific orbital energy relative to that pair.
def make_ejection_event(i):
    """Create a terminal event for body i becoming unbound and distant."""

    def ejection_event(t, state):
        positions = state[:6].reshape(3, 2)
        velocities = state[6:].reshape(3, 2)

        others = [j for j in range(3) if j != i]
        j, k = others

        pair_mass = masses[j] + masses[k]
        pair_position = (
            masses[j] * positions[j] + masses[k] * positions[k]
        ) / pair_mass
        pair_velocity = (
            masses[j] * velocities[j] + masses[k] * velocities[k]
        ) / pair_mass

        displacement = positions[i] - pair_position
        relative_velocity = velocities[i] - pair_velocity
        distance = np.linalg.norm(displacement)

        specific_energy = (
            0.5 * np.dot(relative_velocity, relative_velocity)
            - G * pair_mass / distance
        )

        distance_condition = distance - ejection_radius
        energy_scale = G * MASS_SCALE / LENGTH_SCALE
        energy_condition = specific_energy / energy_scale

        return min(distance_condition / LENGTH_SCALE, energy_condition)

    ejection_event.terminal = True
    ejection_event.direction = 1
    return ejection_event


# Construct all six possible termination events: three pairwise collisions,
# followed by the possible ejection of each of the three bodies. The ordering
# is also used by run_simulation when identifying the event that stopped a run.
collision_events = [
    make_collision_event(0, 1),
    make_collision_event(0, 2),
    make_collision_event(1, 2),
]
ejection_events = [
    make_ejection_event(0),
    make_ejection_event(1),
    make_ejection_event(2),
]
events = collision_events + ejection_events


Total Mechanical Energy

In [7]:
# ============================================================
# CELL 7 — TOTAL MECHANICAL ENERGY
# ============================================================

# Calculate the total kinetic energy of all three bodies and add the pairwise
# gravitational potential energies. This quantity is used to monitor numerical
# energy conservation and solver drift.
def calculate_energy(positions, velocities):
    """Calculate total kinetic plus gravitational potential energy."""

    kinetic = 0.5 * np.sum(
        masses[None, :, None] * velocities**2,
        axis=(1, 2)
    )

    potential = np.zeros(len(positions))

    for i in range(3):
        for j in range(i + 1, 3):
            separation = np.linalg.norm(
                positions[:, j] - positions[:, i],
                axis=1
            )
            potential -= G * masses[i] * masses[j] / separation

    return kinetic + potential


Run One Numerical Simulation

In [8]:
# ============================================================
# CELL 8 — RUN ONE NUMERICAL SIMULATION
# ============================================================

# Integrate one set of initial conditions, stop if a collision or ejection event
# occurs, sample the continuous solution, and calculate energy diagnostics.
# The returned dictionary stores the trajectory, solver metadata, and outcome.
def run_simulation(initial_positions, initial_velocities,
                   rtol=1e-10, atol=1e-12, num_points=5000):
    """Integrate one three-body trajectory and calculate diagnostics."""

    positions0, velocities0 = make_initial_conditions(
        initial_positions, initial_velocities
    )
    initial_state = np.concatenate([positions0.ravel(), velocities0.ravel()])

    solution = solve_ivp(
        derivatives,
        (t_start, t_end),
        initial_state,
        method="DOP853",
        rtol=rtol,
        atol=atol,
        events=events,
        dense_output=True,
        max_step=0.1 * TIME_SCALE
    )

    if not solution.success:
        raise RuntimeError(solution.message)

    event_status = "Undetected collision or ejection"
    event_time = None
    event_names = [
        "Collision: Bodies 1 and 2",
        "Collision: Bodies 1 and 3",
        "Collision: Bodies 2 and 3",
        "Ejection: Body 1",
        "Ejection: Body 2",
        "Ejection: Body 3",
    ]

    for i, event_times in enumerate(solution.t_events):
        if len(event_times) > 0:
            event_status = event_names[i]
            event_time = event_times[0]
            break

    actual_end = solution.t[-1]
    times = np.linspace(t_start, actual_end, num_points)
    sampled_state = solution.sol(times)

    trajectories = sampled_state[:6].T.reshape(-1, 3, 2)
    velocities = sampled_state[6:].T.reshape(-1, 3, 2)

    energy = calculate_energy(trajectories, velocities)
    initial_energy = energy[0]
    energy_scale = G * MASS_SCALE**2 / LENGTH_SCALE

    if abs(initial_energy) < 1e-14 * energy_scale:
        relative_energy_error = np.full_like(energy, np.nan)
        max_energy_error = np.nan
    else:
        relative_energy_error = (energy - initial_energy) / abs(initial_energy)
        max_energy_error = np.max(np.abs(relative_energy_error))

    return {
        "times": times,
        "positions": trajectories,
        "velocities": velocities,
        "energy": energy,
        "relative_energy_error": relative_energy_error,
        "max_energy_error": max_energy_error,
        "initial_energy": initial_energy,
        "function_evaluations": solution.nfev,
        "initial_positions": initial_positions.copy(),
        "initial_velocities": initial_velocities.copy(),
        "rtol": rtol,
        "atol": atol,
        "event_status": event_status,
        "event_time": event_time,
        "actual_end": actual_end,
        "solution": solution,
    }


Baseline Simulation

In [9]:
# ============================================================
# CELL 9 — BASELINE SIMULATION
# ============================================================

# Run the primary initial-value problem using rtol and atol from Cell 3. The
# resulting trajectory is used for the animation and the primary diagnostics.
baseline = run_simulation(
    initial_positions=initial_positions,
    initial_velocities=initial_velocities,
    rtol=rtol,
    atol=atol,
    num_points=num_frames
)

print("\nBASELINE SIMULATION")
print(f"Initial total energy: {baseline['initial_energy']:.6e} J")
print(f"Maximum relative energy drift: {baseline['max_energy_error']:.6e}")
print(f"Derivative evaluations: {baseline['function_evaluations']}")
print(f"Simulation outcome: {baseline['event_status']}")

if baseline["event_time"] is not None:
    print(f"Event time: {baseline['event_time']:.3f} s")

print(
    f"Actual simulated duration: "
    f"{baseline['actual_end'] / TIME_SCALE:.3f} "
    "characteristic time units"
)



BASELINE SIMULATION
Initial total energy: -2.631860e+42 J
Maximum relative energy drift: 1.619109e-08
Derivative evaluations: 56840
Simulation outcome: Undetected collision or ejection
Actual simulated duration: 250.000 characteristic time units


Numerical Convergence Check

In [10]:
# ============================================================
# CELL 10 — NUMERICAL CONVERGENCE CHECK
# ============================================================

# Repeat the same initial-value problem with progressively tighter solver
# tolerances. First compare maximum relative energy drift, then compare adjacent
# trajectories on a shared time grid to assess sensitivity to solver accuracy.
convergence_runs = []

for rtol_test, atol_test in tolerances:
    result = run_simulation(
        initial_positions=initial_positions,
        initial_velocities=initial_velocities,
        rtol=rtol_test,
        atol=atol_test,
        num_points=5000
    )
    convergence_runs.append(result)

print("\nNUMERICAL CONVERGENCE")

convergence_table = pd.DataFrame([
    {
        "rtol": f"{result['rtol']:.0e}",
        "atol": f"{result['atol']:.0e}",
        "Max energy drift": f"{result['max_energy_error']:.6e}",
    }
    for result in convergence_runs
])

display(convergence_table)


# Compare neighboring tolerance levels. Interpolate both numerical solutions
# onto the same time grid and calculate maximum and RMS trajectory separation.
print("\nTRAJECTORY DIFFERENCES")

for coarse, fine in zip(convergence_runs[:-1], convergence_runs[1:]):
    common_end = min(coarse["actual_end"], fine["actual_end"])
    comparison_times = np.linspace(t_start, common_end, 5000)

    coarse_positions = (
        coarse["solution"].sol(comparison_times)
        [:6].T.reshape(-1, 3, 2)
    )
    fine_positions = (
        fine["solution"].sol(comparison_times)
        [:6].T.reshape(-1, 3, 2)
    )

    difference = (coarse_positions - fine_positions) / LENGTH_SCALE
    pointwise_error = np.linalg.norm(difference, axis=2)

    print(
        f"{coarse['rtol']:.0e} vs {fine['rtol']:.0e}: "
        f"Maximum = {np.max(pointwise_error):.6e}, "
        f"RMS = {np.sqrt(np.mean(pointwise_error**2)):.6e}"
    )



NUMERICAL CONVERGENCE


,rtol,atol,Max energy drift
0,1e-07,1e-09,3.075206e-05
1,1e-09,1e-11,2.233367e-07
2,1e-11,1e-13,1.625183e-09



TRAJECTORY DIFFERENCES
1e-07 vs 1e-09: Maximum = 1.236415e-02, RMS = 1.480899e-03
1e-09 vs 1e-11: Maximum = 8.231292e-05, RMS = 9.560818e-06


Create Simulation Title

In [11]:
# ============================================================
# CELL 11 — CREATE SIMULATION TITLE
# ============================================================

# Build a descriptive title from the characteristic scales, body masses, and
# dimensionless initial positions and velocities. This title is printed here
# and passed to the animation function in the next cell.
def make_simulation_title(initial_positions, initial_velocities):
    """Format masses, dimensionless initial conditions, and SI scales."""

    title_lines = [
        "Three-Body Gravitational Simulation",
        f"Scales: L_0 = {LENGTH_SCALE:.3e} m  |  "
        f"V_0 = {VELOCITY_SCALE:.3e} m/s",
    ]

    for i in range(3):
        x, y = initial_positions[i]
        vx, vy = initial_velocities[i]
        title_lines.append(
            f"Body {i + 1}: m = {masses[i]:.3e} kg  |  "
            f"r = ({x:.3g}, {y:.3g}) × L_0  |  "
            f"v = ({vx:.3g}, {vy:.3g}) × V_0"
        )

    return "\n".join(title_lines)


simulation_title = make_simulation_title(
    initial_positions,
    initial_velocities
)

print("\nSIMULATION TITLE")
print(simulation_title)



SIMULATION TITLE
Three-Body Gravitational Simulation
Scales: L_0 = 1.000e+07 m  |  V_0 = 2.583e+06 m/s
Body 1: m = 1.000e+30 kg  |  r = (-1, 0) × L_0  |  v = (0.25, -0.25) × V_0
Body 2: m = 1.000e+30 kg  |  r = (1, 0) × L_0  |  v = (-0.25, 0.25) × V_0
Body 3: m = 1.000e+30 kg  |  r = (0, 7.5) × L_0  |  v = (-0.7, 0) × V_0


Create 2D Animation

In [12]:
# ============================================================
# CELL 12 — CREATE 2D ANIMATION
# ============================================================

# Calculate the unweighted geometric center at every sampled time. The camera
# uses this point as its moving origin, so the animation follows the geometric
# center rather than the center of mass.
def calculate_geometric_center(positions):
    """Return the unweighted geometric center at each time sample."""

    return np.mean(positions, axis=1)


# Create the animation from the baseline trajectory. Positions are recentered
# on the geometric center and displayed in units of 10^6 meters. The animation
# includes moving body markers, trajectory trails, a simulation clock, and title.
def create_animation(baseline, simulation_title):
    """Create the geometric-center-following 2D trajectory animation."""

    times = baseline["times"]
    positions = baseline["positions"]

    geometric_center = calculate_geometric_center(positions)
    positions_relative = positions - geometric_center[:, None, :]
    positions_display = positions_relative / 1e6

    maximum_extent = max(
        3 * LENGTH_SCALE / 1e6,
        1.15 * np.max(np.abs(positions_display))
    )

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(-maximum_extent, maximum_extent)
    ax.set_ylim(-maximum_extent, maximum_extent)
    ax.set_aspect("equal")
    ax.set_xlabel(r"X ($10^6$ m)")
    ax.set_ylabel(r"Y ($10^6$ m)")

    fig.suptitle(simulation_title, fontsize=8, y=0.99)
    fig.subplots_adjust(top=0.80)
    ax.grid(True)

    colors = ["red", "blue", "green"]
    body_markers = []
    body_trails = []

    for i in range(3):
        marker, = ax.plot(
            [], [], "o", color=colors[i], markersize=9,
            label=f"Body {i + 1}"
        )
        trail, = ax.plot(
            [], [], color=colors[i], linewidth=1, alpha=0.6
        )
        body_markers.append(marker)
        body_trails.append(trail)

    ax.legend(loc="upper right")
    time_text = ax.text(0.02, 0.96, "", transform=ax.transAxes)

    def initialize():
        for marker, trail in zip(body_markers, body_trails):
            marker.set_data([], [])
            trail.set_data([], [])
        time_text.set_text("")
        return body_markers + body_trails + [time_text]

    def update(frame):
        for i in range(3):
            x = positions_display[:frame + 1, i, 0]
            y = positions_display[:frame + 1, i, 1]
            body_markers[i].set_data([x[-1]], [y[-1]])
            body_trails[i].set_data(x, y)

        current_time = times[frame]
        time_text.set_text(
            f"Time: {current_time / TIME_SCALE:.2f} characteristic units"
        )
        return body_markers + body_trails + [time_text]

    animation = FuncAnimation(
        fig,
        update,
        frames=len(times),
        init_func=initialize,
        interval=1000 / fps,
        blit=True
    )
    plt.close(fig)
    return animation


animation = create_animation(baseline, simulation_title)
print("Animation created successfully.")


Animation created successfully.


Export and Display Video

In [13]:
# ============================================================
# CELL 13 — EXPORT AND DISPLAY VIDEO
# ============================================================

# Choose the MP4 filename. The video is saved in the notebook's current
# working directory. fps is defined in Cell 3; bitrate and dpi control the
# encoded video quality and rendered resolution.
video_filename = "three_body_simulation.mp4"

writer = FFMpegWriter(
    fps=fps,
    bitrate=1800
)

# Export the animation object created in Cell 12.
animation.save(
    video_filename,
    writer=writer,
    dpi=150
)

print(f"Video saved: {video_filename}")

# Embed the exported animation in the notebook output for preview.
display(HTML(animation.to_html5_video()))

print("\nVIDEO EXPORT COMPLETE")
print(f"Saved: {video_filename}")
print(f"Simulation outcome: {baseline['event_status']}")
print(
    f"Maximum relative energy drift: "
    f"{baseline['max_energy_error']:.6e}"
)


Video saved: three_body_simulation.mp4



VIDEO EXPORT COMPLETE
Saved: three_body_simulation.mp4
Simulation outcome: Undetected collision or ejection
Maximum relative energy drift: 1.619109e-08
